# HAIEnd 23.05 — The Four Operational Personalities

**Same concept as the HAI analysis — applied to the extended HAIEnd dataset (226 sensors, water treatment focus).**
All four files are 100% Normal/Benign, yet each captures a different operational scenario.

| File | Identity | Key trait |
|---|---|---|
| **train1** | Bimodal Focus (2 dominant states) | 17 PP04 states but 76% time at one level |
| **train2** | Richest Diversity | 83 PP04 states — broadest operational coverage |
| **train3** | Completely Frozen | **1 PP04 state only** — system locked at a single setpoint |
| **train4** | Low-Flow Water Treatment Mode | 50 PP04 states, but DM-FT02 flow at **half** the normal rate |

> HAIEnd focuses on the **water treatment and extended P1 sub-processes** where the DM (Dissolved Matter / control) tags live.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path

HAIEND_DIR = Path(r'C:\Users\ahmma\Desktop\farah\haiend-23.05')

COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
SHORT  = ['train1', 'train2', 'train3', 'train4']
LABELS = ['train1\n(Bimodal Focus)', 'train2\n(Rich Diversity)',
          'train3\n(Completely Frozen)', 'train4\n(Low-Flow Mode)']

sns.set_theme(style='whitegrid', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})

print('Loading HAIEnd files...')
dfs = {}
for i in range(1, 5):
    df = pd.read_csv(HAIEND_DIR / f'end-train{i}.csv', parse_dates=['Timestamp'])
    df.set_index('Timestamp', inplace=True)
    dfs[i] = df
    hrs = len(df) * 4 / 3600
    print(f'  train{i}: {len(df):>7,} rows = {hrs:.0f} hrs | {df.shape[1]} sensors')

print('\nAll files loaded.')

---
## Plot 1 — PP04-SP-OUT: How Many Scenarios Did Each File Experience?

`PP04-SP-OUT` is the HMM-driven pump setpoint — each unique value = a distinct operating scenario.
The contrast here is even more extreme than HAI: **train3 had only 1 scenario. train2 had 83.**

In [ ]:
SP_KEY = 'PP04-SP-OUT'

fig = plt.figure(figsize=(18, 9))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.35)
top_axes = [fig.add_subplot(gs[0, j]) for j in range(4)]
bot_ax   = fig.add_subplot(gs[1, :])

state_counts = []
for j, i in enumerate(range(1, 5)):
    ax  = top_axes[j]
    vc  = dfs[i][SP_KEY].value_counts().sort_index()
    state_counts.append(len(vc))

    ax.bar(range(len(vc)), vc.values / 1000,
           color=COLORS[j], alpha=0.85, edgecolor='white', linewidth=0.4)

    top_idx = vc.values.argmax()
    dominant_pct = vc.values[top_idx] / vc.values.sum() * 100
    ax.set_title(f'train{i} — {len(vc)} unique states\nDominant: {vc.index[top_idx]:.2f}  ({dominant_pct:.0f}% of time)',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('State (sorted by setpoint value)', fontsize=8)
    ax.set_ylabel('Duration (k timesteps)', fontsize=8)
    ax.set_xticks([])

    # Special annotation for train3 (completely frozen)
    if i == 3:
        ax.text(0.5, 0.5, 'SINGLE\nSTATE ONLY\n27.92',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=13, fontweight='bold', color=COLORS[2],
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLORS[2], alpha=0.9))

# Bottom comparison bar
bars = bot_ax.bar(SHORT, state_counts, color=COLORS, edgecolor='black', linewidth=0.8, width=0.5)
bot_ax.bar_label(bars, labels=[f'{n} state{"s" if n>1 else ""}' for n in state_counts],
                 padding=5, fontsize=12, fontweight='bold')
bot_ax.set_ylim(0, max(state_counts) * 1.25)
bot_ax.set_ylabel('Number of unique PP04-SP-OUT states')
bot_ax.set_title('How many distinct HMM pump scenarios did each file experience?', fontweight='bold', fontsize=12)

# Reference line: train1 HAI had 15
bot_ax.axhline(15, color='gray', linestyle=':', linewidth=1.2, alpha=0.6)
bot_ax.text(3.6, 16, 'HAI train1\nhad 15', fontsize=8, color='gray', ha='right')

for bar, label in zip(bars, LABELS):
    bot_ax.text(bar.get_x() + bar.get_width()/2, -7,
                label.replace('\n','\n'), ha='center', va='top', fontsize=9, color='#333')

fig.suptitle('PP04-SP-OUT — HMM Pump Setpoint States per Training File\n'
             'train3 is completely locked at ONE setpoint value (27.92)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print('train1: 17 | train2: 83 | train3: 1 | train4: 50')
print('train3 never moved from 27.92 — zero operational diversity in this pump setpoint.')

---
## Plot 2 — train4's Flow Anomaly: DM-FT02 Runs at Half the Normal Rate

In [ ]:
FLOW_SENSORS = ['DM-FT01', 'DM-FT02', 'DM-FT03']
FLOW_LABELS  = ['DM-FT01\n(Primary Flow)', 'DM-FT02\n(Secondary Flow)', 'DM-FT03\n(Tertiary Flow)']

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, sensor, flabel in zip(axes, FLOW_SENSORS, FLOW_LABELS):
    means = [dfs[i][sensor].mean() for i in range(1, 5)]
    stds  = [dfs[i][sensor].std()  for i in range(1, 5)]

    bars = ax.bar(SHORT, means, yerr=stds, color=COLORS,
                  edgecolor='black', linewidth=0.8, width=0.5,
                  error_kw=dict(elinewidth=1.5, capsize=5, capthick=1.5, ecolor='black'))
    ax.bar_label(bars, labels=[f'{v:.0f}' for v in means],
                 padding=8, fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(means) * 1.4)
    ax.set_title(flabel, fontweight='bold')
    ax.set_ylabel('Mean flow value')

    # Highlight outlier
    outlier_idx = int(np.argmax(np.abs(np.array(means) - np.mean(means))))
    ax.get_children()[outlier_idx].set_edgecolor('red')
    ax.get_children()[outlier_idx].set_linewidth(2.5)
    ax.text(outlier_idx, means[outlier_idx] * 0.5,
            'Different!', ha='center', fontsize=9, color='red', fontweight='bold',
            rotation=90)

# Specific annotation for DM-FT02 train4
ax_ft2 = axes[1]
ft2_means = [dfs[i]['DM-FT02'].mean() for i in range(1,5)]
avg_others = np.mean([ft2_means[0], ft2_means[1], ft2_means[2]])
ax_ft2.annotate(f'train4 = {ft2_means[3]:.0f}\n({ft2_means[3]/avg_others*100:.0f}% of avg)',
                xy=(3, ft2_means[3]), xytext=(2.2, ft2_means[3] + 200),
                fontsize=9, color=COLORS[3],
                arrowprops=dict(arrowstyle='->', color=COLORS[3], lw=1.5))

fig.suptitle('Flow Sensor Comparison — train4 DM-FT02 is at ~50% of other files\n'
             'Water treatment running in low-throughput mode',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'DM-FT02 means — train1:{ft2_means[0]:.0f}  train2:{ft2_means[1]:.0f}  train3:{ft2_means[2]:.0f}  train4:{ft2_means[3]:.0f}')

---
## Plot 3 — HMM Setpoint Activity: Same Pattern as HAI, Same Conclusion

In [ ]:
SP18 = '1003.18-OUT'

sp_stats = []
for i in range(1, 5):
    s   = dfs[i][SP18]
    hrs = len(dfs[i]) * 4 / 3600
    chg = int((s.diff().abs() > 1e-6).sum())
    sp_stats.append({
        'file': f'train{i}', 'hours': hrs,
        'changes': chg, 'rate_hr': chg/hrs,
        'range': s.max()-s.min(), 'min': s.min(), 'max': s.max()
    })

sp_df = pd.DataFrame(sp_stats)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel A: changes per hour
ax = axes[0]
bars = ax.bar(sp_df.file, sp_df.rate_hr, color=COLORS, edgecolor='black', linewidth=0.8, width=0.5)
ax.bar_label(bars, labels=[f'{v:.3f}/hr' for v in sp_df.rate_hr], padding=4, fontsize=10, fontweight='bold')
ax.set_ylim(0, sp_df.rate_hr.max() * 1.45)
ax.set_ylabel('Setpoint changes per hour')
ax.set_title(f'1003.18-OUT\nOperator Activity Rate', fontweight='bold')
ax.annotate('Most active\noperator', xy=(0, sp_df.rate_hr.iloc[0]),
            xytext=(0.5, sp_df.rate_hr.max()*1.3), fontsize=8.5, color=COLORS[0],
            arrowprops=dict(arrowstyle='->', color=COLORS[0]))
ax.annotate('Quietest', xy=(2, sp_df.rate_hr.iloc[2]),
            xytext=(1.6, sp_df.rate_hr.max()*0.5), fontsize=8.5, color=COLORS[2],
            arrowprops=dict(arrowstyle='->', color=COLORS[2]))

# Panel B: setpoint range width
ax = axes[1]
bars2 = ax.bar(sp_df.file, sp_df['range'], color=COLORS, edgecolor='black', linewidth=0.8, width=0.5)
ax.bar_label(bars2, labels=[f'{v:.2f}' for v in sp_df['range']], padding=4, fontsize=10, fontweight='bold')
ax.set_ylim(0, sp_df['range'].max() * 1.4)
ax.set_ylabel('Setpoint range (max-min)')
ax.set_title('Setpoint Exploration Width\ntrain3 barely moved (range = 0.26)', fontweight='bold')

# Add interval annotations
for j, (_, row) in enumerate(sp_df.iterrows()):
    ax.text(j, sp_df['range'].max()*0.05, f'[{row["min"]:.1f}\n–{row["max"]:.1f}]',
            ha='center', va='bottom', fontsize=7.5, color='#333')

# Panel C: rolling density timeline
ax = axes[2]
WINDOW = 500
for i, color in enumerate(COLORS, start=1):
    s       = dfs[i][SP18]
    changed = (s.diff().abs() > 1e-6).astype(float)
    density = changed.rolling(WINDOW).mean() * 100
    x       = np.linspace(0, 1, len(density))
    ax.plot(x, density.values, color=color, linewidth=0.85, alpha=0.9, label=f'train{i}')
    mean_d = float(density.mean())
    ax.axhline(mean_d, color=color, linestyle='--', linewidth=1.0, alpha=0.5)

ax.set_xlabel('Normalised time (0=start, 1=end)')
ax.set_ylabel('Change density (%, rolling 500-step)')
ax.set_title('Setpoint Activity Timeline', fontweight='bold')
ax.legend(fontsize=9)

fig.suptitle('HMM Setpoint Activity (1003.18-OUT) — Consistent with HAI Analysis\n'
             'train1 most active, train3 most constrained — the same ordering as in HAI',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot 4 — train1's Bimodal Dominance vs train2's Rich Spread

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

for j, i in enumerate(range(1, 5)):
    ax    = axes.flat[j]
    color = COLORS[j]
    vc    = dfs[i][SP_KEY].value_counts().sort_index()

    # Bar chart of state occupancy
    ax.bar(range(len(vc)), vc.values / 1000,
           color=color, alpha=0.8, edgecolor='white', linewidth=0.3,
           width=1.0)

    # Overlay: cumulative entropy idea — label top-2 states for train1
    n_states  = len(vc)
    total     = vc.values.sum()
    top2_pct  = vc.values[::-1][:2].sum() / total * 100  # top 2 by count

    if i == 1:
        ax.axhspan(vc.values.max()*0.9/1000, vc.values.max()/1000, color='gold', alpha=0.4)
        ax.text(0.98, 0.92, f'Top 2 states\n= {top2_pct:.0f}% of total time',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=9.5, color='darkorange', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', edgecolor='darkorange', alpha=0.8))

    if i == 2:
        ax.text(0.5, 0.92, f'{n_states} states spread\nacross full range',
                transform=ax.transAxes, ha='center', va='top',
                fontsize=9.5, color=color, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', edgecolor=color, alpha=0.8))

    if i == 3:
        ax.text(0.5, 0.5, f'SINGLE POINT\n27.92 only',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=14, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=color, alpha=0.9))

    if i == 4:
        ax.text(0.98, 0.92, f'{n_states} states,\nwider range than train1',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=9.5, color=color, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', edgecolor=color, alpha=0.8))

    ax.set_title(f'train{i} — {LABELS[j].split(chr(10))[1]}',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('PP04-SP-OUT State (sorted by value)', fontsize=9)
    ax.set_ylabel('Duration (k timesteps)', fontsize=9)
    ax.set_xticks([])
    ax.text(0.02, 0.92, f'{n_states} states', transform=ax.transAxes,
            fontsize=12, fontweight='bold', color=color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.8))

fig.suptitle('PP04-SP-OUT Occupancy Distribution — Each File Has a Unique Signature\n'
             'train1 concentrates time in 2 states; train2 spreads evenly across 83',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot 5 — The Physics Never Changes: Perfectly Correlated Signal Pairs (r = 1.0000)

> **The key argument for the digital twin:** HAIEnd has many redundant sensor pairs that are **perfectly correlated (r = 1.0000) in ALL four files**.
> Operating scenarios differ, but the underlying physical laws are constant.

In [ ]:
# Physics pairs: perfect or near-perfect correlation across ALL files
PHYSICS_PAIRS = [
    ('1001.16-OUT', '1001.17-OUT', 'PCV loop: setpoint mirror'),
    ('1002.9-OUT',  '1002.20-OUT', 'Flow rate: redundant channels'),
    ('DM-FCV01-D',  '1003.29-OUT', 'Flow valve: command = feedback'),
    ('DM-TIT01',    'DM-TWIT-04',  'Temperature probes: same location'),
]

fig, axes = plt.subplots(len(PHYSICS_PAIRS), 4, figsize=(18, 11))

rng = np.random.default_rng(42)
for row, (cx, cy, desc) in enumerate(PHYSICS_PAIRS):
    for col, i in enumerate(range(1, 5)):
        ax    = axes[row, col]
        color = COLORS[col]
        df    = dfs[i]
        idx   = rng.choice(len(df), min(2500, len(df)), replace=False)
        x, y  = df[cx].iloc[idx].values, df[cy].iloc[idx].values
        mask  = np.isfinite(x) & np.isfinite(y)
        x, y  = x[mask], y[mask]

        r_val, _ = stats.pearsonr(x, y)

        ax.scatter(x, y, alpha=0.12, s=3, color=color, rasterized=True)
        m, b = np.polyfit(x, y, 1)
        xf   = np.linspace(x.min(), x.max(), 100)
        ax.plot(xf, m*xf + b, color='black', linewidth=2.0)

        if row == 0:
            ax.set_title(f'train{i}', fontweight='bold', fontsize=11)
        ax.set_xlabel(cx, fontsize=7)
        ax.set_ylabel(cy, fontsize=7)
        ax.text(0.05, 0.93, f'r = {r_val:.4f}', transform=ax.transAxes,
                fontsize=11, fontweight='bold',
                color='black' if abs(r_val) < 0.95 else 'darkgreen',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9))

    # Row label on left
    axes[row, 0].set_ylabel(f'{cy}\n({desc})', fontsize=8)

fig.suptitle(
    'HAIEnd Physics Pairs — Correlation is Constant Across All 4 Files\n'
    'Operating scenarios change. Physical laws do not.\n'
    'The digital twin learns the laws — not just the numbers.',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## Plot 6 — Radar Chart: HAIEnd Operational Fingerprints

In [ ]:
metrics = [
    ('PP04 states\n(PP04-SP-OUT)',      lambda df: df['PP04-SP-OUT'].nunique()),
    ('Secondary Flow\n(DM-FT02)',       lambda df: df['DM-FT02'].mean()),
    ('Boiler Pressure\n(DM-PIT01)',     lambda df: df['DM-PIT01'].mean()),
    ('Setpoint Activity\n(1003.18/hr)', lambda df: (df['1003.18-OUT'].diff().abs()>1e-6).sum()/(len(df)*4/3600)),
    ('Setpoint Range\n(1003.18)',       lambda df: df['1003.18-OUT'].max()-df['1003.18-OUT'].min()),
    ('Water Level\n(DM-LIT01)',         lambda df: df['DM-LIT01'].mean()),
]

metric_names = [m[0] for m in metrics]
raw    = np.array([[fn(dfs[i]) for _, fn in metrics] for i in range(1, 5)])
normed = (raw - raw.min(axis=0)) / (raw.max(axis=0) - raw.min(axis=0) + 1e-9)

N      = len(metric_names)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, (ax_radar, ax_table) = plt.subplots(1, 2, figsize=(16, 7))
fig.delaxes(ax_radar)
ax_radar = fig.add_subplot(121, polar=True)

for j, (color, label) in enumerate(zip(COLORS, SHORT)):
    vals = normed[j].tolist() + normed[j].tolist()[:1]
    ax_radar.plot(angles, vals, color=color, linewidth=2.5, label=label)
    ax_radar.fill(angles, vals, color=color, alpha=0.12)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(metric_names, fontsize=9)
ax_radar.set_yticklabels([])
ax_radar.set_title('HAIEnd Operational Fingerprint\n(normalised 0-1)', fontweight='bold', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)

# Summary table
ax_table.axis('off')
personalities = [
    ['train1', '17', '1,098', '0.43/hr', '2.84', 'Bimodal: 76% in\n2 dominant states'],
    ['train2', '83', '1,017', '0.25/hr', '2.05', 'Richest diversity\n— 83 unique states'],
    ['train3', '1',  '1,214', '0.13/hr', '0.26', 'Frozen at one\nsetpoint (27.92)'],
    ['train4', '50', '613',   '0.22/hr', '2.19', 'Low-flow mode\n(FT02 at ~50%)'],
]
col_headers = ['File', 'PP04\nStates', 'DM-FT02\nMean', 'SP Activity\n(1003.18)', 'SP Range', 'Identity']
table = ax_table.table(
    cellText=personalities, colLabels=col_headers,
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.2)
for j in range(len(col_headers)):
    table[0, j].set_facecolor('#2C3E50')
    table[0, j].set_text_props(color='white', fontweight='bold')
for row_i, color in enumerate(COLORS, start=1):
    for col_j in range(len(col_headers)):
        table[row_i, col_j].set_facecolor(color + '33')

ax_table.set_title('Summary: HAIEnd Four Operational Personalities', fontweight='bold', fontsize=12, pad=15)

fig.suptitle('HAIEnd 23.05 — Same "Normal", Four Different Operational Worlds',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Plot 7 — Time Series Snapshot: See the Differences in Real Time

In [ ]:
N_SHOW  = 5000
SENSORS = [
    ('PP04-SP-OUT', 'PP04 Pump Setpoint'),
    ('DM-FT02',     'DM-FT02 Secondary Flow'),
    ('1003.18-OUT', '1003.18 HMM Setpoint'),
]

fig, axes = plt.subplots(len(SENSORS), 4, figsize=(20, 9), sharey='row')

for row, (sensor, ylabel) in enumerate(SENSORS):
    for col, i in enumerate(range(1, 5)):
        ax   = axes[row, col]
        vals = dfs[i][sensor].iloc[:N_SHOW].values
        ax.plot(vals, color=COLORS[col], linewidth=0.6, alpha=0.9)
        ax.fill_between(range(len(vals)), vals, alpha=0.12, color=COLORS[col])
        if row == 0:
            ax.set_title(f'train{i}\n{LABELS[col].split(chr(10))[1]}',
                         fontweight='bold', fontsize=10)
        if col == 0:
            ax.set_ylabel(ylabel, fontsize=9)
        ax.set_xlabel('Timestep', fontsize=8)
        mean_v = float(np.nanmean(vals))
        ax.axhline(mean_v, color='black', linestyle='--', linewidth=1.0, alpha=0.6)
        ax.text(N_SHOW*0.98, mean_v, f' {mean_v:.1f}',
                va='center', ha='right', fontsize=7.5, color='black')

fig.suptitle(f'Time Series — First {N_SHOW:,} Timesteps of Each File\n'
             'PP04: train3 is a flat line (frozen). DM-FT02: train4 is clearly lower.',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Final Conclusion

```
╔══════════════════════════════════════════════════════════════════════════╗
║  WHAT THE HAIEnd DATA PROVES                                            ║
╠══════════════════════════════════════════════════════════════════════════╣
║  train1  │ 17 PP04 states, but 76% time in just 2 of them.            ║
║          │ Highest setpoint activity (0.43 changes/hr).               ║
╠══════════╪═════════════════════════════════════════════════════════════╣
║  train2  │ 83 PP04 states — the broadest operational coverage.        ║
║          │ Best file for learning the full range of normal behaviour.  ║
╠══════════╪═════════════════════════════════════════════════════════════╣
║  train3  │ 1 PP04 state. System locked at 27.92. Range = 0.26.        ║
║          │ Ideal for steady-state / baseline modelling.                ║
╠══════════╪═════════════════════════════════════════════════════════════╣
║  train4  │ 50 PP04 states, DM-FT02 at ~50% of normal flow.            ║
║          │ Water treatment running in low-throughput / conservation.   ║
╠══════════════════════════════════════════════════════════════════════════╣
║  PHYSICS: 1001.16-OUT vs 1001.17-OUT → r = 1.0000 in EVERY file.      ║
║  DM-FCV01-D vs 1003.29-OUT → r = 1.0000 in EVERY file.                ║
║  The physical coupling is invariant — only the operating point shifts. ║
║  The digital twin learns LAWS, not noise.                              ║
╚══════════════════════════════════════════════════════════════════════════╝
```

### Cross-dataset consistency check
| Pattern | HAI | HAIEnd |
|---|---|---|
| Most active file | train1 (0.42/hr) | train1 (0.43/hr) |
| Quietest file | train3 (0.13/hr) | train3 (0.13/hr) |
| Most diverse HMM | train1 (15 states) | train2 (83 states) |
| Most frozen | train3 (4 states) | train3 (**1 state**) |
| Unique outlier | train1 low RPM | train4 low flow |

> The setpoint activity ordering (**train1 > train2 > train4 > train3**) is **identical** in both datasets,
> confirming these represent the same four operational regimes seen from different sensor suites.